# Segment Ink Detection (PHercParis4)

Trains a 3D→2D CNN on 11 labelled segments of PHercParis4 (Scroll 1).
Works on Kaggle and local systems — environment is auto-detected.

Labels are pseudo-labels (outputs of a prior `tile64-stride16` model); loss has an ignore-band on ambiguous label values.

## 1. Data setup (auto-detects Kaggle vs local)

In [ ]:
import os, ssl, sys, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

IS_KAGGLE = os.path.exists('/kaggle/input') or os.path.exists('/kaggle/working')
print(f'Environment: {"Kaggle" if IS_KAGGLE else "local"}')

# Segments sorted smallest → largest (surface volume sizes in GB)
ALL_SEGMENTS = [
    ('20231221180251', 3.5), ('20231031143852', 3.7), ('20231016151002', 4.0),
    ('20231106155351', 4.5), ('20230702185753', 4.6), ('20231210121321', 5.1),
    ('20230929220926', 6.1), ('20231022170901', 6.3), ('20231005123336', 8.6),
    ('20231012184424', 9.8), ('20231007101619', 14.2),
]

# On Kaggle: /kaggle/working is capped at ~20 GB. Pick smallest 3 by default.
if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data/labelled_segments')
    SEGMENTS_TO_USE = [s for s, _ in ALL_SEGMENTS[:3]]  # ~11 GB total
else:
    DATA_ROOT = Path('../data/labelled_segments')
    SEGMENTS_TO_USE = [s for s, _ in ALL_SEGMENTS]  # use all locally

print(f'Data root: {DATA_ROOT}')
print(f'Segments selected: {len(SEGMENTS_TO_USE)}')

In [ ]:
# S3 download (only runs on Kaggle or if local data missing)

BASE_URL = 'https://data.aws.ash2txt.org/samples/PHercParis4/segments'
VOLUME   = '7.91um-54keV-volume-20230205180739.tifs'
NUM_LAYERS = 33

LABEL_FN = ('PHercParis4-{seg}-7.91um-54keV-volume-20230205180739-'
            '20230702185753-tile64-stride16.tif')

_ctx = ssl.create_default_context()
_ctx.check_hostname = False
_ctx.verify_mode = ssl.CERT_NONE

def _fetch(url, dest, retries=3):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 1000:
        return True
    for i in range(retries):
        try:
            req = urllib.request.Request(url)
            with urllib.request.urlopen(req, timeout=60, context=_ctx) as r, open(dest, 'wb') as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
            return True
        except Exception as e:
            print(f'  retry {i+1} for {dest.name}: {e}')
    return False

def download_segment(seg):
    d = DATA_ROOT / seg
    label_url = f'{BASE_URL}/{seg}/ink-detection/' + LABEL_FN.format(seg=seg)
    _fetch(label_url, d / 'ink_labels.tif')
    with ThreadPoolExecutor(max_workers=6) as pool:
        futs = [pool.submit(_fetch,
                f'{BASE_URL}/{seg}/surface-volumes/{VOLUME}/{i:02d}.tif',
                d / 'surface_volume' / f'{i:02d}.tif')
                for i in range(NUM_LAYERS)]
        for _ in as_completed(futs): pass
    ok = len(list((d / 'surface_volume').glob('*.tif'))) == NUM_LAYERS and (d / 'ink_labels.tif').exists()
    print(f'  {seg}: {"OK" if ok else "INCOMPLETE"}')
    return ok

need_download = [s for s in SEGMENTS_TO_USE
                 if not ((DATA_ROOT / s / 'ink_labels.tif').exists()
                         and len(list((DATA_ROOT / s / 'surface_volume').glob('*.tif'))) == NUM_LAYERS)]

if need_download:
    print(f'Downloading {len(need_download)} segments...')
    for seg in need_download:
        download_segment(seg)
else:
    print('All selected segments already present — skipping download.')

## 2. Imports & config

In [ ]:
# Ensure this notebook's directory is on path (so segment_model.py imports work)
if IS_KAGGLE:
    # On Kaggle, upload segment_model.py alongside this notebook or as a utility script
    sys.path.insert(0, '/kaggle/working')
    sys.path.insert(0, '/kaggle/input')
else:
    sys.path.insert(0, '..')
    sys.path.insert(0, '../src')

import numpy as np, torch, tifffile as tf
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from segment_model import (
    CFG, SegmentInkNet, SegmentStreamDataset,
    masked_soft_bce_dice, fbeta, predict_segment,
    load_segment_label, derive_mask, load_segment_volume,
)

CFG['val_segment'] = SEGMENTS_TO_USE[0]  # smallest → fast validation
train_segs = [s for s in SEGMENTS_TO_USE if s != CFG['val_segment']]
print(f'Train segments: {train_segs}')
print(f'Val segment:    {CFG["val_segment"]}')
print(f'Device: {CFG["device"]}')
print(f'CFG: {CFG}')

## 3. Datasets

In [ ]:
train_ds = SegmentStreamDataset(
    DATA_ROOT, train_segs,
    patch_size=CFG['patch_size'], patches_per_seg=CFG['patches_per_seg'],
    ink_pos_thresh=CFG['ink_pos_thresh'], ink_neg_thresh=CFG['ink_neg_thresh'],
    augment=True, shuffle_segments=True,
)
val_ds = SegmentStreamDataset(
    DATA_ROOT, [CFG['val_segment']],
    patch_size=CFG['patch_size'], patches_per_seg=80,
    ink_pos_thresh=CFG['ink_pos_thresh'], ink_neg_thresh=CFG['ink_neg_thresh'],
    augment=False, shuffle_segments=False,
)

# num_workers=0 on Kaggle (IterableDataset + multiprocessing is fragile); local: same.
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], num_workers=0, pin_memory=True)
print('Datasets ready')

## 4. Model + optimizer

In [ ]:
device = CFG['device']
model = SegmentInkNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['num_epochs'])
scaler = torch.amp.GradScaler(enabled=(device == 'cuda'))
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {n_params/1e6:.2f}M params on {device}')

## 5. Training loop (AMP + gradient accumulation)

In [ ]:
from pathlib import Path
MODEL_DIR = Path('/kaggle/working/models' if IS_KAGGLE else 'models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

hist = {'train_loss': [], 'val_loss': [], 'val_f05': []}
best_f05 = -1.0

for epoch in range(CFG['num_epochs']):
    model.train()
    total, n = 0.0, 0
    opt.zero_grad()
    for i, (img, y, m) in enumerate(train_loader):
        img = img.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)
        m   = m.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=(device == 'cuda')):
            logits = model(img)
            loss = masked_soft_bce_dice(logits, y, m,
                      ignore_low=CFG['ignore_band'][0], ignore_high=CFG['ignore_band'][1]
                  ) / CFG['grad_accum']
        scaler.scale(loss).backward()
        if (i + 1) % CFG['grad_accum'] == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad()
        total += loss.item() * CFG['grad_accum']; n += 1
        if n % 50 == 0:
            print(f'  epoch {epoch+1} step {n} loss={total/n:.4f}')
    sched.step()
    train_loss = total / max(n, 1)

    # validation
    model.eval()
    vloss = vf = vn = 0.0
    with torch.no_grad():
        for img, y, m in val_loader:
            img, y, m = img.to(device), y.to(device), m.to(device)
            with torch.amp.autocast(device_type='cuda', enabled=(device == 'cuda')):
                logits = model(img)
                vloss += masked_soft_bce_dice(logits, y, m).item()
            vf += fbeta(logits, y, m, thresh=CFG['threshold']); vn += 1
    val_loss = vloss / max(vn, 1); val_f05 = vf / max(vn, 1)
    hist['train_loss'].append(train_loss); hist['val_loss'].append(val_loss); hist['val_f05'].append(val_f05)
    print(f'[epoch {epoch+1}] train={train_loss:.4f}  val={val_loss:.4f}  F0.5={val_f05:.4f}')

    if val_f05 > best_f05:
        best_f05 = val_f05
        torch.save(model.state_dict(), MODEL_DIR / 'best_segment_model.pth')
        print(f'  ✓ saved best (F0.5={best_f05:.4f})')

print(f'Done. Best F0.5 = {best_f05:.4f}')

## 6. Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['train_loss'], label='train'); axes[0].plot(hist['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('epoch')
axes[1].plot(hist['val_f05']); axes[1].set_title('Val F0.5'); axes[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 7. Inference on val segment

In [ ]:
PRED_DIR = Path('/kaggle/working/predictions' if IS_KAGGLE else 'predictions')
PRED_DIR.mkdir(exist_ok=True, parents=True)

model.load_state_dict(torch.load(MODEL_DIR / 'best_segment_model.pth', map_location=device))
seg_id = CFG['val_segment']
prob = predict_segment(
    model, DATA_ROOT / seg_id,
    patch_size=CFG['patch_size'], stride=CFG['patch_size'] // 2,
    device=device, amp=True,
)
np.save(PRED_DIR / f'{seg_id}_prob.npy', prob)
tf.imwrite(PRED_DIR / f'{seg_id}_prob.tif', (prob * 255).astype(np.uint8))
print(f'Saved prediction for {seg_id}  shape={prob.shape}  mean={prob.mean():.3f}')

## 8. Visualization — CT / prediction / ground truth / overlay

In [ ]:
seg_dir = DATA_ROOT / seg_id
gt  = load_segment_label(seg_dir)
ct  = tf.imread(str(seg_dir / 'surface_volume' / '16.tif')).astype(np.float32) / 255.0  # middle slice
mask = derive_mask(load_segment_volume(seg_dir))

binary = (prob > CFG['threshold']).astype(np.uint8) * mask

# Optional: crop to a readable region for display
H, W = prob.shape
ys, xs = np.where(mask)
y0, y1 = ys.min(), ys.max(); x0, x1 = xs.min(), xs.max()

fig, ax = plt.subplots(2, 3, figsize=(16, 10))
ax[0,0].imshow(ct[y0:y1, x0:x1], cmap='gray'); ax[0,0].set_title('CT slice (z=16)')
ax[0,1].imshow(prob[y0:y1, x0:x1], cmap='viridis'); ax[0,1].set_title('Prediction (prob)')
ax[0,2].imshow(gt[y0:y1, x0:x1], cmap='gray'); ax[0,2].set_title('Pseudo-label')
ax[1,0].imshow(binary[y0:y1, x0:x1], cmap='gray'); ax[1,0].set_title(f'Binary (t={CFG["threshold"]})')
ov = np.stack([ct[y0:y1,x0:x1]]*3, -1)
ov[..., 0] = np.maximum(ov[..., 0], binary[y0:y1,x0:x1])  # red overlay
ax[1,1].imshow(ov); ax[1,1].set_title('Prediction over CT')
ax[1,2].imshow(np.abs(prob[y0:y1,x0:x1] - gt[y0:y1,x0:x1]), cmap='hot'); ax[1,2].set_title('|pred − label|')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.savefig(PRED_DIR / f'{seg_id}_vis.png', dpi=100); plt.show()

## Next steps

- Run `python src/experiment_filters.py --pred predictions/{seg_id}_prob.npy` to extract letter candidates and score them against Greek uppercase templates.
- Try training on more segments: edit `SEGMENTS_TO_USE` in cell 1.
- Soft-label threshold and ignore-band live in `CFG` — tune them against validation F0.5.